# 🧠 BrainStack — fine-tuning our own answering model

**Step 2 of `docs/FINETUNE_PLAN.md`.** Runs on a **free Colab T4 GPU**.

### What this notebook does, in one line

It shows a small model (**Qwen 2.5 3B**) about a thousand examples of *"here
are numbered passages, here is the ideal cited answer"* until it copies the
habit.

```
 train.jsonl  ──►  Qwen 2.5 3B (4-bit)  +  LoRA  ──►  adapter  ──►  GGUF
   ~1,200          the base model          the         ~60 MB      ~2 GB
   examples        someone else trained    part we                 runs on
                                           train                   your laptop
```

### Before you press anything

1. **Runtime → Change runtime type → T4 GPU.** Free tier gives you one most of
   the time. If it says none are available, come back in an hour.
2. Upload `training/data/train.jsonl` and `val.jsonl` — either to Google Drive
   (recommended, survives a crash) or straight into this session.
3. **Do not close the tab.** A free Colab session dies after ~90 minutes idle.

Total run time: roughly **40–70 minutes** of training after ~20 minutes of setup.


---
## A · Check we actually got a GPU

If this cell errors or prints nothing, you are on a CPU runtime and training
would take days. Fix the runtime type before going on.


In [ ]:
!nvidia-smi


## B · Install Unsloth

**Unsloth** is a library that makes LoRA fine-tuning ~2× faster and use much
less memory. It is the reason a 3B model fits on a free T4 at all.

This takes 3–5 minutes. Ignore the dependency warnings — Colab always prints
them.


In [ ]:
%%capture
!pip install -q unsloth
# Pull the latest nightly too: Unsloth moves fast and the released wheel on
# Colab is often a few weeks behind on model support.
!pip install -q --upgrade --no-deps --force-reinstall git+https://github.com/unslothai/unsloth.git


## C · Get the data in

**Option 1 (recommended) — Google Drive.** Put `train.jsonl` and `val.jsonl`
in a folder called `brainstack` in your Drive. Survives a session crash.

**Option 2 — direct upload.** Faster to start, gone when the session dies.

Run ONE of the two cells below.


In [ ]:
# OPTION 1 — Google Drive
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/brainstack'
SAVE_DIR = '/content/drive/MyDrive/brainstack'   # checkpoints survive a crash


In [ ]:
# OPTION 2 — upload from your laptop (skip if you used Drive)
# from google.colab import files
# files.upload()          # pick train.jsonl, then run again for val.jsonl
# DATA_DIR = '/content'
# SAVE_DIR = '/content'


## D · Load and look at the data

Never train on data you have not looked at. This cell prints the split, the
answerable/refusal balance, and one full example — read it.


In [ ]:
import json, os

def load(name):
    path = os.path.join(DATA_DIR, name)
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train_rows = load('train.jsonl')
val_rows   = load('val.jsonl')

from collections import Counter
kinds = Counter(r['kind'] for r in train_rows)
print(f"train {len(train_rows)}   val {len(val_rows)}")
print(f"answerable {kinds['answerable']}   refusal traps {kinds['refusal']} "
      f"({kinds['refusal']/len(train_rows):.0%})")

print("\n" + "="*70)
ex = train_rows[0]
print(ex['user'][:900])
print("-"*70)
print("IDEAL ANSWER:", ex['assistant'])


## E · Load the base model

`Qwen/Qwen2.5-3B-Instruct` — 3 billion numbers, already trained by Alibaba on
a huge amount of text. It already speaks English and follows instructions. It
does **not** yet have our citing/refusing habit. That is what we add.

`load_in_4bit=True` squeezes those numbers down so the whole thing fits in the
T4's 16 GB alongside the training machinery. That is the **Q** in QLoRA.

Why Qwen and not Llama 3.2: Apache 2.0 licence, and it is not gated on
HuggingFace — no token, no forms, nothing to break when a session restarts.


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048   # our rows measure ~500-900 tokens; cell G verifies it
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,      # None = pick the best for this GPU
    load_in_4bit   = True,
)
print("loaded:", BASE_MODEL)


## F · Attach the LoRA adapter

Here is the whole idea of LoRA in one picture:

```
   the base model            what we train
   3,000,000,000 numbers  +  ~30,000,000 numbers   (about 1%)
   ❄️  FROZEN                🔥 LEARNING
```

We do not touch the 3 billion numbers. We bolt on a small extra layer and
train only that. It is ~100× cheaper, and the result is a 60 MB file instead
of a 6 GB one.

- `r = 16` — how much room the new layer has to learn. The standard start.
- `lora_alpha = 16` — how loudly the new layer speaks. Match it to `r`.
- `target_modules` — which parts of the model get an adapter. These seven are
  the usual full set; fewer learns less.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    lora_alpha                 = 16,
    lora_dropout               = 0,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",   # trades a little speed for lots of memory
    random_state               = 42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"training {trainable:,} of {total:,} parameters  ({trainable/total:.2%})")


## G · Put the rows into Qwen's chat format

Our JSONL has three plain fields: `system`, `user`, `assistant`. Qwen expects
its own markup with `<|im_start|>` tags around each turn. `apply_chat_template`
does that conversion — using the tokenizer's own template, so it is exactly
what the model saw during its original training.

Then we check token lengths **before** training. If rows are longer than
`MAX_SEQ_LENGTH` they get silently cut off, the model never sees the end of its
own training answers, and you find out an hour later. Cheap check, expensive
mistake.


In [ ]:
from datasets import Dataset

def to_text(row):
    return tokenizer.apply_chat_template(
        [
            {"role": "system",    "content": row["system"]},
            {"role": "user",      "content": row["user"]},
            {"role": "assistant", "content": row["assistant"]},
        ],
        tokenize=False,
    )

train_ds = Dataset.from_list([{"text": to_text(r)} for r in train_rows])
val_ds   = Dataset.from_list([{"text": to_text(r)} for r in val_rows])

lengths = sorted(len(tokenizer(t["text"])["input_ids"]) for t in train_ds)
p50, p95, mx = lengths[len(lengths)//2], lengths[int(len(lengths)*0.95)], lengths[-1]
print(f"tokens per row —  p50 {p50}   p95 {p95}   max {mx}   (limit {MAX_SEQ_LENGTH})")
over = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
if over:
    print(f"\n⚠ {over} rows ({over/len(lengths):.1%}) would be TRUNCATED.")
    print("  Raise MAX_SEQ_LENGTH to 3072 in cell E and re-run from there.")
else:
    print("\n✓ nothing gets truncated.")

print("\n" + "="*70)
print(train_ds[0]["text"][:600])


## H · Set up the trainer

One setting here matters more than all the others: **`train_on_responses_only`**.

```
  <|im_start|>system   You are BrainStack ...     ← don't learn this
  <|im_start|>user     Context: [1] ... Question: ← don't learn this
  <|im_start|>assistant Claims are due in 45 days [1].   ← LEARN THIS
```

Without it, the model spends most of its effort learning to *predict the
context passages we already give it* — wasted, and it dilutes the actual
lesson. With it, the loss only counts the answer.


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,     # effective batch = 8
        warmup_steps                = 10,
        num_train_epochs            = 2,
        learning_rate               = 2e-4,
        logging_steps               = 5,
        eval_strategy               = "steps",
        eval_steps                  = 25,     # watch val loss: overfitting alarm
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 42,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)

# Mask everything before the assistant's turn — Qwen's ChatML markers.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)
print("ready")


## I · Train

Watch two numbers:

| | meaning | what you want |
|---|---|---|
| `loss` | how wrong it is on data it is learning from | goes down, then flattens |
| `eval_loss` | how wrong it is on data it has **never seen** | goes down too |

```
 GOOD                          OVERFITTING (stop / fewer epochs)
 loss ●●●●●●●●●●●●             loss     ●●●●●●●●●●●●
 eval ●●●●●●●●●●●●             eval ●●●●●●        ●●●●●●●
      ────────────►                  ─────────────────────►
 both fall together           eval turns back UP = memorising
```

If `eval_loss` starts climbing, the model is memorising our 1,200 examples
instead of learning the habit. Fix: set `num_train_epochs = 1` and re-run.

**40–70 minutes.** Leave the tab open.


In [ ]:
stats = trainer.train()
print(stats)


## J · The loss curve

Screenshot this. It is the single most useful picture to have in an interview —
it shows you watched the training instead of just running it.


In [ ]:
import matplotlib.pyplot as plt

hist  = trainer.state.log_history
train = [(h['step'], h['loss'])      for h in hist if 'loss' in h]
evals = [(h['step'], h['eval_loss']) for h in hist if 'eval_loss' in h]

plt.figure(figsize=(9, 4))
plt.plot(*zip(*train), label='train loss', linewidth=2)
if evals:
    plt.plot(*zip(*evals), label='eval loss (unseen data)', marker='o', linewidth=2)
plt.xlabel('step'); plt.ylabel('loss')
plt.title('BrainStack 3B — grounded answering')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

if evals:
    print(f"final train {train[-1][1]:.4f}   final eval {evals[-1][1]:.4f}")
    if len(evals) > 1 and evals[-1][1] > min(e[1] for e in evals) * 1.05:
        print("⚠ eval loss rose from its best — try num_train_epochs = 1")


## K · Sanity check — does it actually cite?

Before spending 40 minutes on export, ask it two questions from the **held-out**
val set (it has never seen these):

1. an answerable one → expect a short answer with `[n]` markers
2. a refusal trap → expect the exact refusal sentence, nothing else

If it rambles, invents facts, or cites `[9]` when there are 5 passages, do not
export. Go back and look at the data.


In [ ]:
FastLanguageModel.for_inference(model)

def answer(row):
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": row["system"]},
         {"role": "user",   "content": row["user"]}],
        tokenize=False, add_generation_prompt=True,
    )
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=300, temperature=0.2, do_sample=True)
    return tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

for kind in ("answerable", "refusal"):
    row = next((r for r in val_rows if r["kind"] == kind), None)
    if not row: continue
    print("="*70)
    print(f"[{kind}]  {row['user'].rsplit('Question: ', 1)[-1]}")
    print(f"\n  OURS   : {answer(row).strip()}")
    print(f"  IDEAL  : {row['assistant']}")


## L · Save the adapter

60 MB. Useless on its own — it only works plugged into Qwen 2.5 3B. Save it to
Drive so a dead session costs you nothing.


In [ ]:
ADAPTER_DIR = f"{SAVE_DIR}/brainstack-3b-lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved →", ADAPTER_DIR)


---
# Step 3 · EXPORT — make it run on your laptop

Everything below is **Step 3 of the plan**. Two things happen:

```
  LoRA adapter (60 MB)  +  base model (6 GB)
            │
            ▼   merge      — glue the adapter into the weights
      merged model  (~6 GB)
            │
            ▼   quantize   — squeeze to 4 bits
   brainstack-3b-q4_k_m.gguf  (~2 GB)   ← this is what you download
```

**Q4_K_M** is the sweet spot: ~2 GB, runs in your 7.7 GB of RAM next to the
backend and a browser, and loses very little quality on a task this narrow.

This cell takes **20–45 minutes**. It builds `llama.cpp` from source the first
time.


In [ ]:
model.save_pretrained_gguf(
    f"{SAVE_DIR}/brainstack-3b-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)
print("done")


In [ ]:
# Find the .gguf and check its size
import glob, os
for p in glob.glob(f"{SAVE_DIR}/brainstack-3b-gguf/**/*.gguf", recursive=True):
    print(f"{os.path.getsize(p)/1e9:.2f} GB   {p}")


## Getting the file to your laptop

**From Drive (recommended):** it is already in your Drive folder — open
drive.google.com and download it there. Resumable, unlike the cell below.

**Direct download:** uncomment and run. A 2 GB Colab download that fails at 90%
starts over, so only use this on a solid connection.


In [ ]:
# from google.colab import files
# files.download(f"{SAVE_DIR}/brainstack-3b-gguf/unsloth.Q4_K_M.gguf")


---
## ✅ Done here — what happens next

1. Put the `.gguf` next to `training/Modelfile` in the repo.
2. Install Ollama, then:
   ```bash
   cd training
   python make_modelfile.py          # writes Modelfile from the live prompt
   ollama create brainstack-3b -f Modelfile
   ollama run brainstack-3b          # paste a context block, check it cites
   ```
3. Point the backend at it:
   ```
   LLM_PROVIDER=local
   ```
4. Run the comparison — **Step 5**:
   ```bash
   python eval/run_eval.py --notes "brainstack-3b (finetuned)"
   ```

See `training/README.md`.
